# Creating Finetuning Datasets

In [ ]:
import datasets
import pandas as pd
from datasets import load_dataset
from collections import Counter
import spacy, re
import random
import csv
from sklearn.model_selection import train_test_split

## Existing Approximate Domain Data

#### AjayMukundS/Indian_Legal_NER_Dataset -- COURT, DATE, ORG, GPE, PROVISION + PER as combination

In [13]:
text = "In this reference under Section 66 (1) of the Indian Income-tax Act, 1922, at the instance of the assessee Messrs. Dayabhai & Co. of Barwani, the question posed for our answer is: Whether on the facts and in the circumstances of this case, the assessee is entitled to registration under Section 26-A of the Indian Income-tax Act for the assessment year 1956-57?"
ent_markings = [
{
"end": 38,
"label": "PROVISION",
"start": 24
},   
{
"end": 73,
"label": "STATUTE",
"start": 46
},
{
"end": 140,
"label": "ORG",
"start": 115
},
{
"end": 302,
"label": "PROVISION",
"start": 290
},
{
"end": 331,
"label": "STATUTE",
"start": 310
}
]

In [15]:
dataset = load_dataset("AjayMukundS/Indian_Legal_NER_Dataset")
nlp = spacy.load("en_core_web_trf")

for example in dataset["train"]:
    text = example["text"]
    for ent in example["entities"]:
        span_text = text[ent["start"]:ent["end"]]
        word_count = len(span_text.split())
        if word_count > 20:
            print(f"{ent['label']:<20} ({word_count} words): {span_text[:80]}")

PRECEDENT            (21 words): M. Swaminathan v. The Chairman and Managing Director, Tamil Nadu Small Industrie
PRECEDENT            (23 words): Oriental Insurance Co. Ltd. v. Deo Patodi [(2009) 13 SCC 123 : (2009) 5 SCC (Civ
CASE_NUMBER          (24 words): Appeal Nos. 801 of 1993, 798 of 1993, 800 of 1993, 797 of 1993, 794 of 1993, 796
PRECEDENT            (30 words): State of Haryana   Vs.   Anil Kumar, 2004 (1)  

 Punj. LR 69 , 
               
PRECEDENT            (21 words): Sanjay Singh and another Vs. U.P. Public Service Commission, Allahabad and anoth
CASE_NUMBER          (21 words): C. Ws. Nos. 1203 to 1207 of 1958, and 17, 124, 202 and 206 to 209, 509 and 510 o
PRECEDENT            (24 words): Uttam Singh Duggal & Co. Ltd. v. United Bank of India, reported as VI (2000) SLT
PRECEDENT            (22 words): State of Maharashtra vs. Kanchanmala Vijaysing Shirke, 1995 ACJ 1021 (SC) = (199
STATUTE              (29 words): Kerala State Road Transport Corporation (Passenger Grou

In [ ]:
def tokenize_with_offsets_spacy(text):
    """
    this function takes text as input, tokenizes it, and returns the token's text, its index in the text, and its final character index
    """
    doc = nlp.make_doc(text) #tokenize text
    return [
        (tok.text, tok.idx, tok.idx + len(tok.text)) #return the text, token index, token's ending index
        for tok in doc      #for each token in the stripped text
        if tok.text.strip()]


def char_spans_to_bio(tokens_with_offsets, entities):
    """
    this function assigns a BIO label given a token's character offsets and 
    the existing span indexes presented in the Indian Legal NER dataset that it overlaps with

    """
    labels = ["O"] * len(tokens_with_offsets) #everything starts with O

    sorted_ents = sorted(entities, key=lambda e: e["start"]) #sort entities by start so B- is assigned to the first overlapping token

    for ent in sorted_ents:
        e_start = ent["start"] #assign the start
        e_end   = ent["end"] #assign the end
        label   = ent.get("label") #extract the label

        first = True
        for i, (tok, t_start, t_end) in enumerate(tokens_with_offsets):
            if t_end > e_start and t_start < e_end: # overlap condition: token and entity spans intersect
                labels[i] = f"B-{label}" if first else f"I-{label}" #make sure to assign B to the first token and I to following
                first = False

    return labels #now we have a bio+nerc label tag for a token!

def normalize_token(tok):
    """
    this function cleans any unnecessary whitespace from the text
    """
    tok = re.sub(r'\s+', ' ', tok)  # collapse internal whitespace/newlines
    return tok.strip()

def write_conll(dataset_split, out_path):
    """
    this functiion writes the text entity pairs to conll
    """
    with open(out_path, "w", encoding="utf-8") as infile:
        for example in dataset_split: #we get the text, entities pairs
            text     = example.get("text", "")
            entities = example.get("entities", [])
            if not text or not text.strip(): #skip any empty texts
                continue
            tokens_with_offsets = tokenize_with_offsets_spacy(text) #get token offsets!
            if not tokens_with_offsets:
                continue
            labels = char_spans_to_bio(tokens_with_offsets, entities) #get the labels per token
            for (token, index, end), lbl in zip(tokens_with_offsets, labels):
                clean_tok = normalize_token(token)
                if not clean_tok:          # skip if token was empty
                    continue
                infile.write(f"{clean_tok}\t{lbl}\n")
            infile.write("\n")

In [ ]:
for split_name in dataset.keys():
    out_file = f"indian_legal_ner_{split_name}_converted.conll"
    write_conll(dataset[split_name], out_file) 

first_split = list(dataset.keys())[0]
print(f"\nPreview of indian_legal_ner_{first_split}_converted.conll:\n")
with open(f"indian_legal_ner_{first_split}_converted.conll", encoding="utf-8") as infile:
    for i, line in enumerate(f):
        if i >= 20:
            break
        print(line, end="")      


Preview of indian_legal_ner_train_converted.conll:

(	O
7	O
)	O
On	O
specific	O
query	O
by	O
the	O
Bench	O
about	O
an	O
entry	O
of	O
Rs	O
.	O
1,31,37,500	O
on	O
deposit	O
side	O
of	O


### Initial Counts

In [ ]:
def read_conll(path):
    """
    shows more accurate reading of the dataframe than default read_csv
    """
    return pd.read_csv(
        path, sep='\t', header=None, names=["token","label"],
        quoting=csv.QUOTE_NONE, keep_default_na=False, na_filter=False,
        skip_blank_lines=True,
        engine='python',
    )

In [52]:
# read_conll("indian_legal_ner_train_attempt_1.conll")["label"].value_counts().sort_index()
read_conll("indian_legal_ner_train_converted.conll")["label"].value_counts().sort_index()

label
B-CASE_NUMBER       1040
B-COURT             2367
B-DATE              1885
B-GPE               1398
B-JUDGE             2325
B-LAWYER            3505
B-ORG               1441
B-OTHER_PERSON      2653
B-PETITIONER        3068
B-PRECEDENT         1351
B-PROVISION         2384
B-RESPONDENT        3862
B-STATUTE           1804
B-WITNESS            881
I-CASE_NUMBER       4786
I-COURT            10264
I-DATE              3292
I-GPE                296
I-JUDGE             2114
I-LAWYER            3271
I-ORG               3002
I-OTHER_PERSON      2112
I-PETITIONER        5582
I-PRECEDENT        14747
I-PROVISION         5761
I-RESPONDENT       11169
I-STATUTE           4121
I-WITNESS            767
O                 506423
Name: count, dtype: int64

In [53]:
read_conll("indian_legal_ner_validation_converted.conll")["label"].value_counts().sort_index()

label
B-CASE_NUMBER       118
B-COURT             282
B-DATE              216
B-GPE               178
B-JUDGE             154
B-LAWYER            506
B-ORG               155
B-OTHER_PERSON      263
B-PETITIONER        181
B-PRECEDENT         173
B-PROVISION         257
B-RESPONDENT        266
B-STATUTE           221
B-WITNESS            58
I-CASE_NUMBER       428
I-COURT            1021
I-DATE              199
I-GPE                47
I-JUDGE             191
I-LAWYER            547
I-ORG               337
I-OTHER_PERSON      187
I-PETITIONER        422
I-PRECEDENT        2258
I-PROVISION         614
I-RESPONDENT        927
I-STATUTE           474
I-WITNESS            56
O                 50884
Name: count, dtype: int64

In [ ]:
mapping = { "B-JUDGE": "B-PERSON", "I-JUDGE": "I-PERSON", "B-LAWYER": "B-PERSON", #mapping to only need our target labels
           "B-OTHER_PERSON": "B-PERSON", "I-OTHER_PERSON": "I-PERSON", 
           "B-WITNESS": "B-PERSON", "I-WITNESS": "I-PERSON", "I-LAWYER": "I-PERSON",

           "B-STATUTE": "B-LAW", "I-STATUTE": "I-LAW", "B-CASE_NUMBER": "O",
             "I-CASE_NUMBER": "O", "B-PRECEDENT": "O", "I-PRECEDENT": "O",
             "B-RESPONDENT": "O", "I-RESPONDENT": "O","B-PETITIONER": "O",
               "I-PETITIONER": "O"
           }

with open("indian_legal_ner_train_converted.conll", "r", encoding = "utf-8") as infile: #use the og conll for base
    lines = infile.readlines()

with open("indian_legal_ner_train_converted_MAPPED.conll", "w", encoding = "utf-8") as outfile: #then write to a new file
    for i, line in enumerate(lines):
        if line.strip() == "":
            outfile.write("\n")
            continue
        token, label = line.strip().split("\t")
        if label == "O":
            outfile.write(f"{token}\tO\n")
            continue
        label = mapping.get(label, label)
        outfile.write(f"{token}\t{label}\n")

with open("indian_legal_ner_validation_converted_MAPPED.conll", "r", encoding = "utf-8") as infile:
    lines = infile.readlines()

with open("Inlegal_validation_FINAL.conll", "w", encoding = "utf-8") as outfile:

    for i, line in enumerate(lines):
        if line.strip() == "":
            outfile.write("\n")
            continue

        token, label = line.strip().split("\t")

        if label == "O":
            outfile.write(f"{token}\tO\n")
            continue

        label = mapping.get(label, label)
        outfile.write(f"{token}\t{label}\n")

# inlegalner_trn_df = read_conll("inlegal_train_FINAL.conll")
inlegal_val_df = read_conll("Inlegal_validation_FINAL.conll")

In [ ]:
read_conll("inlegal_train_FINAL.conll")["label"].value_counts().sort_index()

label
B-COURT          2367
B-DATE           1885
B-GPE            1398
B-LAW            1804
B-ORG            1441
B-PERSON         9364
B-PROVISION      2384
I-COURT         10264
I-DATE           3292
I-GPE             296
I-LAW            4121
I-ORG            3002
I-PERSON         8264
I-PROVISION      5761
O              552028
Name: count, dtype: int64

In [ ]:
read_conll("inlegal_validation_FINAL.conll")["label"].value_counts().sort_index()

label
B-COURT          282
B-DATE           216
B-GPE            178
B-LAW            221
B-ORG            155
B-PERSON         981
B-PROVISION      257
I-COURT         1021
I-DATE           199
I-GPE             47
I-LAW            474
I-ORG            337
I-PERSON         981
I-PROVISION      614
O              55657
Name: count, dtype: int64

## E-NER

first, all.csv -->  blank edgar txt for \n sentence boundaries. imported into excel to be saved as a tsv --> tab_correct_edgar.txt. to remove docstart and other incorrect misc stuff, it becomes clean_tabbed_edgar.txt

In [2]:
edgar_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\E-NER\all.csv"
edgar_df = pd.read_csv(edgar_file, sep=',', header=None)

In [ ]:
counter = 0
with open(edgar_file, "r", encoding="utf-8") as infile:
    lines = infile.readlines()
    for line in lines:
        if line.strip() == ",O":
            counter += 1
        # print(repr(line))
print(counter)

11696


In [ ]:
sentence_boundary = ',O\n' #need to replace these sentence boundaries so theyre uniform with the other conll files

with open(edgar_file, "r", encoding="utf-8") as infile, open("blank_edgar.txt", "w", encoding="utf-8") as outfile:
    for line in infile.readlines():
        # print(repr(line))
        if line == sentence_boundary:
            outfile.write("\n")
        else:
            outfile.write(line)
#now we have a file with the right sentence boundaries

In [ ]:
#for ease, i put the blank_edgar.txt file into excel and exported it back as a tab separated file.
counter = 0
with open("tab_corrected_edgar.txt", "r", encoding="utf-8") as infile: ##put the new line separated file with comma separation into excel, then exported it back in as a tsv!!
    lines = infile.readlines()
    for line in lines:
        if line.strip() == "\t":
            counter += 1
        # print(repr(line))
#
print(counter)
#yay it works

0


In [ ]:
false_doc = 'Column1\tColumn2\n' #getting rid of false column names
false_nl = ',O\n' #and any false new lines
false_comma = '","' #and false commas since it was originally a comma separated file, so commas are in quotations!

with open("tab_correct_edgar.txt", "r", encoding="utf-8") as infile, \
     open("clean_tabbed_edgar.txt", "w", encoding="utf-8") as outfile:
    
    for line in infile:
        if line == false_doc:
            continue
        if line == false_nl:
            outfile.write('\n')
            continue

        parts = line.rstrip("\n").split("\t")
        if len(parts) != 2:
            outfile.write(line)
            continue

        text, label = parts[0], parts[1]

        if text == false_comma:
            outfile.write(f',\t{label}\n')
        elif text.startswith('"') or text.endswith('"'):
            # print(text)
            outfile.write(f'{text[1:-1]}\t{label}\n')
        else:
            outfile.write(f'{text}\t{label}\n')

#now we have a tab separated file!

In [ ]:
with open("clean_tabbed_edgar.txt", "r", encoding="utf-8") as infile:
    lines = infile.readlines()

cleaned = []
for i, line in enumerate(lines):
    if line.strip() == "" and i > 0 and lines[i - 1].strip() == "": #getting rid of further empty lines
        continue
    cleaned.append(line)

with open("clean_oldlabel_edgar.txt", "w", encoding="utf-8") as outfile:
    for line in cleaned:
        if line == "\t\n":
            line = "\n" #replacing the false \t new lines with just \n
        outfile.writelines(line)

In [63]:
for i, line in enumerate(lines[:50]):
    if not line.strip():
        print(f"Line {i}: {repr(line)}")

Line 1: '\n'
Line 7: '\n'
Line 19: '\n'
Line 30: '\n'
Line 42: '\n'


In [61]:
counter = 0
with open("clean_oldlabel_edgar.txt", "r", encoding="utf-8") as infile: ##put the new line separated file with comma separation into excel, then exported it back in as a tsv!!
    lines = infile.readlines()
    for line in lines:
        if line.strip() == "":
            counter += 1
        # print(repr(line))

print(counter)

11696


In [ ]:
### now that we have a cleaned file, we need to replace all the I- entities that start a span with B- since we are using IOB2!!
with open("clean_oldlabel_edgar.txt", "r", encoding="utf-8") as infile:
    test_set = infile.readlines()

with open("cleaned_final_edgar.txt", "w", encoding="utf-8") as outfile:
    for i, line in enumerate(test_set):
        if line.strip() == "":
            outfile.write("\n")
            continue
        token, label = line.strip().split("\t")
        if label == "O":
            outfile.write(f"{token}\tO\n") #keep Os
            continue
        actual_label = label[2:] #extract only the label
        prev_label = ""
        if i > 0 and test_set[i - 1].strip() != "": #
            _, prev_label = test_set[i - 1].strip().split("\t")

        if label.startswith("I-") and not prev_label.endswith(actual_label): #if the current label starts with I and the previous label does not end in the same label,
            outfile.write(f"{token}\tB-{actual_label}\n") #then its a B- token
        else:
            outfile.write(f"{token}\t{label}\n") #otherwise keep the token label pair

In [ ]:
edgar_df[1].value_counts() ##before preprocessing

1
O                    391772
I-BUSINESS            10764
I-MISCELLANEOUS        4783
I-LEGISLATION/ACT      3407
I-LOCATION             1733
I-GOVERNMENT           1334
I-PERSON               1327
I-COURT                 169
B-BUSINESS               28
B-MISCELLANEOUS          20
B-LEGISLATION/ACT        16
B-LOCATION                6
P                         4
B-GOVERNMENT              3
B-PERSON                  2
I-LOC                     1
Name: count, dtype: int64

In [55]:
edited_edgar_file = r"cleaned_final_edgar.txt"
edgar_df_2 = pd.read_csv(edited_edgar_file, sep='\t', header=None)

In [56]:
edgar_df_2[1].value_counts().sort_index()

1
B-BUSINESS             3779
B-COURT                  30
B-GOVERNMENT            580
B-LEGISLATION/ACT      1171
B-LOC                     1
B-LOCATION             1026
B-MISCELLANEOUS        1599
B-PERSON                635
I-BUSINESS             7013
I-COURT                 139
I-GOVERNMENT            757
I-LEGISLATION/ACT      2252
I-LOCATION              713
I-MISCELLANEOUS        3204
I-PERSON                694
O                    380076
P                         4
Name: count, dtype: int64

In [ ]:
mapping = {"B-BUSINESS": "B-ORG", "I-BUSINESS": "I-ORG", "B-LOCATION": "B-GPE", "I-LOCATION": "I-GPE",
           "P": "O", "B-GOVERNMENT": "B-ORG", "I-GOVERNMENT": "I-ORG", "B-LOC": "B-GPE", #mapping to our target labels!
           "B-LEGISLATION/ACT": "B-LAW", "I-LEGISLATION/ACT": "I-LAW", "B-MISCELLANEOUS": "O",
           "I-MISCELLANEOUS": "O"}

with open("cleaned_final_edgar.txt", "r", encoding = "utf-8") as infile:
    lines = infile.readlines()

with open("full_edgar.conll", "w", encoding = "utf-8") as outfile: 
    for i, line in enumerate(lines):
        if line.strip() == "":
            outfile.write("\n") #new sentence boundary
            continue
        token, label = line.strip().split("\t") #separate token, label
        if label == "O":
            outfile.write(f"{token}\tO\n") #keep O
            continue
        label = mapping.get(label, label) #get the mapped label and overwrite current label
        outfile.write(f"{token}\t{label}\n") #now we have a new mapped file!

In [85]:
new_edgar = pd.read_csv(r"full_edgar.conll", sep= "\t", header=None)
new_edgar[1].value_counts().sort_index()

1
B-COURT         30
B-GPE         1027
B-LAW         1171
B-ORG         4359
B-PERSON       635
I-COURT        139
I-GPE          713
I-LAW         2252
I-ORG         7770
I-PERSON       694
O           384883
Name: count, dtype: int64

In [ ]:
with open("full_edgar.conll", encoding="utf-8") as infile:
    for i, line in enumerate(f):
        if line.strip() and '"' in line:
            print(f"Line {i}: {repr(line)}")

In [3]:
#this dataset came with doc boundaries, so we should maintain them.
new_doc = '-DOCSTART-\tO\n'
sentence_boundary = '\n'

document_detected = []
document_sentences = []
current_sentence = []

with open("full_edgar.conll", "r", encoding="utf-8") as infile:
    for line in infile:
        if line == new_doc:
            if current_sentence:
                document_sentences.append(current_sentence)
                current_sentence = []
            if document_sentences:
                document_detected.append(document_sentences)
            document_sentences = []
        elif line == sentence_boundary:
            if current_sentence:
                document_sentences.append(current_sentence)
                current_sentence = []
        else:
            current_sentence.append(line)

    if current_sentence:
        document_sentences.append(current_sentence)
    if document_sentences:
        document_detected.append(document_sentences)

    #now we have documents separated and sentences within them!

In [59]:
len(document_detected[5])
# (document_detected[5]) -- has 159 sentences

159

In [4]:
# document_detected[0] = the first document as a list of lists of sentences split into tokens
entity = "B-COURT"
for i, docs in enumerate(document_detected): # list of all docs -- 52 docs total
    #doc = list of lists of tokens/ list of list of sentences -- split into tokens. so len of doc == len of sentences
    # print(doc)
    for doc in docs:
        for tokens in doc:
            if entity in tokens:
                print(i, tokens)

4 U.S.	B-COURT

4 U.S.	B-COURT

4 U.S.	B-COURT

4 U.S.	B-COURT

4 Eastern	B-COURT

4 California	B-COURT

4 District	B-COURT

4 Second	B-COURT

4 California	B-COURT

4 Ninth	B-COURT

4 Ninth	B-COURT

4 Ninth	B-COURT

4 Ninth	B-COURT

4 Ninth	B-COURT

4 United	B-COURT

5 San	B-COURT

5 United	B-COURT

5 United	B-COURT

37 United	B-COURT

37 United	B-COURT

37 Texas	B-COURT

37 United	B-COURT

37 D.C.	B-COURT

37 United	B-COURT

37 New	B-COURT

37 New	B-COURT

37 New	B-COURT

37 New	B-COURT

50 Court	B-COURT

50 Court	B-COURT



In [ ]:
## use sk learn train test split to split edgar docs in court and non court docs, before combining them for edgar train and validation
## splits
random.seed(38)

def has_entity(doc, entity="B-COURT"):
    for sentence in doc:
        for line in sentence:
            if entity in line:
                return True
    return False


court_docs = []
non_court_docs = []

# split documents into court and non-court groups since its the least populated class + most relevant one for us
for doc in document_detected:
    if has_entity(doc):
        court_docs.append(doc)
    else:
        non_court_docs.append(doc)

print(f"Documents with COURT: {len(court_docs)}")
print(f"Documents without COURT: {len(non_court_docs)}")

# stratified split on each group to make sure court is proportionally distributed
court_train, court_val = train_test_split(court_docs, test_size=0.1, random_state=38)
non_court_train, non_court_val = train_test_split(non_court_docs, test_size=0.1, random_state=38)

# combine
edgar_train = court_train + non_court_train
edgar_val = court_val + non_court_val

random.shuffle(edgar_train)
random.shuffle(edgar_val)

print(f"\nEDGAR train: {len(edgar_train)} docs, {sum(len(d) for d in edgar_train)} sents")
print(f"EDGAR val: {len(edgar_val)} docs, {sum(len(d) for d in edgar_val)} sents")
print(f"COURT docs in train: {len(court_train)}")
print(f"COURT docs in val: {len(court_val)}")

#now we have E-NER train and val splits made on court entity allocation!

Documents with COURT: 4
Documents without COURT: 48

EDGAR train: 46 docs, 10260 sents
EDGAR val: 6 docs, 1384 sents
COURT docs in train: 3
COURT docs in val: 1


In [ ]:
def write_only_edgar(path, edgar_docs):
    ## writing just edgar splits to sep files to do distribution checks
    with open(path, "w", encoding="utf-8") as infile:
        # edgar documents (sentence order preserved within docs)
        for doc in edgar_docs:
            infile.write("-DOCSTART-\tO\n\n")
            for sentence in doc:
                for token_line in sentence:
                    infile.write(token_line if token_line.endswith("\n") else token_line + "\n")
                infile.write("\n")

write_only_edgar("EDGAR_TRAINING.conll", edgar_train)
write_only_edgar("EDGAR_VALIDATION.conll", edgar_val)

In [ ]:
FINAL_COMBINED_FILE = pd.read_csv("EDGAR_TRAINING.conll", sep = "\t", header = None, engine="python",
   on_bad_lines="skip")
FINAL_COMBINED_FILE[1].value_counts().sort_index()

1
B-COURT         15
B-GPE          951
B-LAW         1033
B-ORG         3514
B-PERSON       569
I-COURT         86
I-GPE          652
I-LAW         2006
I-ORG         5504
I-PERSON       625
O           345012
Name: count, dtype: int64

In [ ]:
FINAL_COMBINED_FILE = pd.read_csv("EDGAR_VALIDATION.conll", sep = "\t", header = None, engine="python",
   on_bad_lines="skip")
FINAL_COMBINED_FILE[1].value_counts().sort_index()

1
B-COURT        15
B-GPE          76
B-LAW         138
B-ORG         845
B-PERSON       66
I-COURT        53
I-GPE          61
I-LAW         246
I-ORG        2266
I-PERSON       69
O           39819
Name: count, dtype: int64

In [ ]:
print((edgar_train[0]))


## Combining Indian Legal NER and E-NER for an Approximate Legal Domain Dataset

In [ ]:
indian_train_sents = []
current_sentence = []

with open("inlegal_train_FINAL.conll", "r", encoding="utf-8") as infile: #get all the inlegalner train sentences
    for line in infile:
        if line.strip() == "":
            if current_sentence:
                indian_train_sents.append(current_sentence)
                current_sentence = []
        else:
            current_sentence.append(line)
    if current_sentence:
        indian_train_sents.append(current_sentence)

indian_val_sents = []
current_sentence = []

with open("inlegal_validation_FINAL.conll", "r", encoding="utf-8") as infile:  #get all the inlegalner val sentences
    for line in infile:
        if line.strip() == "":
            if current_sentence:
                indian_val_sents.append(current_sentence)
                current_sentence = []
        else:
            current_sentence.append(line)
    if current_sentence:
        indian_val_sents.append(current_sentence)


def write_conll_file(path, edgar_docs, indian_sents):
    with open(path, "w", encoding="utf-8") as infile:
        #now we write the e-ner data per doc, preserving sentence order within docs
        for doc in edgar_docs:
            infile.write("-DOCSTART-\tO\n\n")
            for sentence in doc:
                for token_line in sentence:
                    infile.write(token_line if token_line.endswith("\n") else token_line + "\n")
                infile.write("\n")
        # and now the indian sentences per doc, no order needed to preserve
        for sentence in indian_sents:
            for token_line in sentence:
                infile.write(token_line if token_line.endswith("\n") else token_line + "\n")
            infile.write("\n")

random.seed(22)
random.shuffle(indian_train_sents) #shuffle them
random.shuffle(indian_val_sents)
random.shuffle(edgar_train) #shuffle these too
random.shuffle(edgar_val)

write_conll_file("OS_TRAINING.conll", edgar_train, indian_train_sents) #write them to 
write_conll_file("OS_VALIDATION.conll", edgar_val, indian_val_sents)    #a combined file

print(f"EDGAR train: {len(edgar_train)} docs, {sum(len(d) for d in edgar_train)} sents")
print(f"EDGAR val: {len(edgar_val)} docs, {sum(len(d) for d in edgar_val)} sents")
print(f"Indian train: {len(indian_train_sents)} sents")
print(f"Indian val: {len(indian_val_sents)} sents")


EDGAR train: 46 docs, 10260 sents
EDGAR val: 6 docs, 1384 sents
Indian train: 10995 sents
Indian val: 1074 sents


In [64]:
read_conll("OS_TRAINING.conll")["label"].value_counts().sort_index()

label
B-COURT          2382
B-DATE           1885
B-GPE            2349
B-LAW            2837
B-ORG            4955
B-PERSON         9933
B-PROVISION      2384
I-COURT         10350
I-DATE           3292
I-GPE             948
I-LAW            6127
I-ORG            8506
I-PERSON         8889
I-PROVISION      5761
O              897086
Name: count, dtype: int64

In [ ]:
read_conll("OS_VALIDATION.conll")["label"].value_counts().sort_index()

label
B-COURT          297
B-DATE           216
B-GPE            254
B-LAW            359
B-ORG           1000
B-PERSON        1047
B-PROVISION      257
I-COURT         1074
I-DATE           199
I-GPE            108
I-LAW            720
I-ORG           2603
I-PERSON        1050
I-PROVISION      614
O              95482
Name: count, dtype: int64